# CINEOS FIRST LIGHT — Real GPU ignition
Run in Google Colab with **Runtime > Change runtime type > GPU**. This notebook fails loudly: SUCCESS produces a real MP4; FAILURE prints the exact traceback.


In [ ]:
!nvidia-smi
import os, sys, subprocess, traceback, json, time
try:
 import torch
 print('torch:', torch.__version__)
 print('CUDA available:', torch.cuda.is_available())
 if not torch.cuda.is_available(): raise RuntimeError('CUDA NOT AVAILABLE — select a GPU runtime in Colab')
 print('GPU:', torch.cuda.get_device_name(0))
 print('CUDA:', torch.version.cuda)
 free,total=torch.cuda.mem_get_info(); print('VRAM free/total GB:', round(free/2**30,2), round(total/2**30,2))
except Exception:
 traceback.print_exc(); raise


In [ ]:
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg safetensors


In [ ]:
# Clone/update the exact CINEOS development branch
%cd /content
!rm -rf cineos-phase2
!git clone -b codex/short-drama-agent-sprint-1 https://github.com/freebook4u2022-967/cineos-phase2.git
%cd /content/cineos-phase2
!git rev-parse HEAD
!pip -q install -e .


In [ ]:
# REAL video-model First Light. CogVideoX-2b is used as the initial open pretrained foundation.
import torch, traceback, os, time, json
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
MODEL='THUDM/CogVideoX-2b'
OUT='/content/cineos_first_light_001.mp4'
PROMPT='Cinematic night street after light rain, reflections on asphalt, a courier motorcycle passes under street lamps, realistic film lighting, smooth motion'
evidence={'model':MODEL,'gpu':torch.cuda.get_device_name(0),'cuda':torch.version.cuda,'status':'STARTED','started':time.time()}
try:
 print('MODEL LOAD START:', MODEL)
 pipe=CogVideoXPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
 pipe.enable_model_cpu_offload()
 print('WEIGHTS LOADED')
 torch.cuda.reset_peak_memory_stats()
 print('INFERENCE START')
 frames=pipe(prompt=PROMPT, num_videos_per_prompt=1, num_inference_steps=25, num_frames=33, guidance_scale=6.0).frames[0]
 export_to_video(frames, OUT, fps=8)
 evidence.update(status='SUCCESS',output=OUT,bytes=os.path.getsize(OUT),peak_vram_gb=round(torch.cuda.max_memory_allocated()/2**30,3),finished=time.time())
 print('🔥 CINEOS FIRST LIGHT:', OUT, os.path.getsize(OUT), 'bytes')
 print(json.dumps(evidence,indent=2))
except Exception as e:
 evidence.update(status='FAILED',error=repr(e),finished=time.time())
 print(json.dumps(evidence,indent=2))
 traceback.print_exc()
 raise


In [ ]:
# Artifact evidence + playback
import os, hashlib
from IPython.display import Video, display
p='/content/cineos_first_light_001.mp4'
assert os.path.exists(p) and os.path.getsize(p)>0
h=hashlib.sha256(open(p,'rb').read()).hexdigest()
print('MP4:',p); print('bytes:',os.path.getsize(p)); print('sha256:',h)
display(Video(p,embed=True))
